# Intialization

In [0]:
import pyspark.sql.functions as f
from pyspark.sql.types import StringType, DateType
from pyspark.sql.functions import col, trim
from pyspark.sql.window import Window

# Read Bronze table

In [0]:
df = spark.table("workspace.bronze.crm_prd_info")
df.display()

# Silver Transformations

### Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

### Product Key Parsing

In [0]:
df = df.withColumn("cat_id", f.regexp_replace(f.substring(col("prd_key"), 1, 5), "-", "_"))
df = df.withColumn("prd_key", f.substring(col("prd_key"), 7, f.length(col("prd_key"))))

In [0]:
df.display()

### Show Nulls

In [0]:
null_filter = " OR ".join([f"{c} IS NULL" for c in df.columns])
df.filter(null_filter).show()

### Cost Cleanup

In [0]:
df = df.withColumn("prd_cost", f.coalesce(col("prd_cost"), f.lit(0)))

In [0]:
null_filter = " OR ".join([f"{c} IS NULL" for c in df.columns])
df.filter(null_filter).show()

In [0]:
df.filter(col("prd_cost").isNull()).show()

### Product Line Normalization

In [0]:
df = (
        df.withColumn(
        "prd_line",
        f.when(f.upper(col("prd_line")) == "M", "Mountain")
        .when(f.upper(col("prd_line")) == "R", "Road")
        .when(f.upper(col("prd_line")) == "S", "Other Sales")
        .when(f.upper(col("prd_line")) == "T", "Touring")
        .otherwise("n/a")
    )
)
        

In [0]:
df.groupBy("prd_line").count().show()

### Date Casting

In [0]:
df = df.withColumn("prd_start_dt", col("prd_start_dt").cast(DateType()))

### Calculate end date as one day before the next start date

In [0]:
window_spec = Window.partitionBy("prd_key").orderBy("prd_start_dt")

df = df.withColumn(
    "prd_end_dt",
    f.date_sub(
        f.lead("prd_start_dt").over(window_spec),
        1
    ).cast("date")
)

### Renaming Columns

In [0]:
RENAME_MAP = {
    "prd_id": "product_id",
    "cat_id": "category_id",
    "prd_key": "product_number",
    "prd_nm": "product_name",
    "prd_cost": "product_cost",
    "prd_line": "product_line",
    "prd_start_dt": "start_date",
    "prd_end_dt": "end_date"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

In [0]:
df.limit(10).display()

# Writing Silver Table

In [0]:
(df.write
 .mode("overwrite")
 .format("delta")
 .saveAsTable("workspace.silver.crm_products"))

# Sanity checks of silver table

In [0]:
%sql
SELECT * FROM workspace.silver.crm_products LIMIT 10